# Threshold-based evaluation

Average precision integrates over the confidence ranking, so a correct
detection that the model ranks low is penalised almost as heavily as a miss.
Because the annotation pipeline thresholded teacher proposals at 0.30, the
boxes the annotator added by hand are exactly the boxes every model ranks
low, and AP on that subset collapses even though recall exceeds 0.96.

Precision, recall, and F1 at a fixed operating point do not depend on
ranking. This notebook reports them across a threshold sweep, selects an
operating point per model, and repeats the teacher-derived versus
human-added split under those metrics.

Needs in Drive: `gold_predictions.json`, `gold_v2_verified.zip`,
`gold_v2_coco_c30.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
import os, json, glob, re
import numpy as np, pandas as pd
from collections import defaultdict

preds_all = json.load(open(f'{DRIVE}/gold_predictions.json'))

!rm -rf /content/gold_ann
!unzip -q {DRIVE}/gold_v2_verified.zip -d /content/gold_ann
ann = glob.glob('/content/gold_ann/**/instances_default.json', recursive=True)[0]
verified = json.load(open(ann))
teacher  = json.load(open(f'{DRIVE}/gold_v2_coco_c30.json'))

def to_xyxy(d):
    names = {im['id']: im['file_name'].split('/')[-1] for im in d['images']}
    out = defaultdict(list)
    for a in d['annotations']:
        x, y, w, h = a['bbox']
        out[names[a['image_id']]].append([x, y, x+w, y+h])
    for n in names.values(): out.setdefault(n, [])
    return dict(out)

gt_all, gt_teach = to_xyxy(verified), to_xyxy(teacher)
print('models:', list(preds_all), '| tiles:', len(gt_all))

In [ ]:
def iou_matrix(A, B):
    if len(A) == 0 or len(B) == 0:
        return np.zeros((len(A), len(B)), np.float32)
    A = np.asarray(A, np.float32); B = np.asarray(B, np.float32)
    ix1 = np.maximum(A[:,0,None], B[None,:,0]); iy1 = np.maximum(A[:,1,None], B[None,:,1])
    ix2 = np.minimum(A[:,2,None], B[None,:,2]); iy2 = np.minimum(A[:,3,None], B[None,:,3])
    inter = np.clip(ix2-ix1,0,None) * np.clip(iy2-iy1,0,None)
    aA = ((A[:,2]-A[:,0])*(A[:,3]-A[:,1]))[:,None]
    aB = ((B[:,2]-B[:,0])*(B[:,3]-B[:,1]))[None,:]
    u = aA + aB - inter
    return np.divide(inter, u, out=np.zeros_like(inter), where=u>0)

def prf(preds, gtd, thr, iou_t=0.5, keys=None):
    """Greedy matching at a fixed confidence threshold. No ranking involved."""
    keys = keys if keys is not None else list(gtd)
    TP = FP = FN = 0
    for t in keys:
        g = gtd.get(t, [])
        p = sorted([b for b in preds.get(t, []) if b[4] >= thr], key=lambda b: -b[4])
        if not p:
            FN += len(g); continue
        if not g:
            FP += len(p); continue
        M = iou_matrix([b[:4] for b in p], g)
        taken = set(); tp = 0
        for i in range(len(p)):
            for cand in np.argsort(-M[i]):
                if M[i, cand] < iou_t: break
                if cand not in taken:
                    taken.add(int(cand)); tp += 1; break
        TP += tp; FP += len(p) - tp; FN += len(g) - tp
    P = TP / max(TP + FP, 1); R = TP / max(TP + FN, 1)
    F = 2 * P * R / max(P + R, 1e-9)
    return dict(P=round(P,4), R=round(R,4), F1=round(F,4), TP=TP, FP=FP, FN=FN)

## Threshold sweep

In [ ]:
rows = []
for name, preds in preds_all.items():
    for thr in np.arange(0.05, 0.75, 0.05):
        m = prf(preds, gt_all, float(thr))
        rows.append(dict(model=name, conf=round(float(thr),2), **m))
sweep = pd.DataFrame(rows)
for name, g in sweep.groupby('model'):
    print(f'\n{name}')
    print(g[['conf','P','R','F1']].to_string(index=False))
sweep.to_csv(f'{DRIVE}/results_sweep.csv', index=False)

In [ ]:
# Operating point maximising F1
best = sweep.loc[sweep.groupby('model').F1.idxmax()].set_index('model')
print(best[['conf','P','R','F1']].to_string())
best.to_csv(f'{DRIVE}/results_operating_point.csv')

## Teacher-derived versus human-added, under threshold metrics

The comparison that collapsed under AP, recomputed at each model's
operating point.

In [ ]:
MATCH = 0.5
gt_kept, gt_added = {}, {}
for t, boxes in gt_all.items():
    tb = gt_teach.get(t, [])
    if not boxes:
        gt_kept[t] = gt_added[t] = []
        continue
    M = iou_matrix(boxes, tb)
    matched = M.max(1) >= MATCH if M.shape[1] else np.zeros(len(boxes), bool)
    gt_kept[t]  = [b for b, m in zip(boxes, matched) if m]
    gt_added[t] = [b for b, m in zip(boxes, matched) if not m]

rows = []
for name, preds in preds_all.items():
    thr = float(best.loc[name, 'conf'])
    for label, gtd in [('all', gt_all), ('teacher-derived', gt_kept),
                       ('human-added', gt_added)]:
        keys = [t for t in gtd if gtd[t]] if label != 'all' else list(gt_all)
        m = prf(preds, gtd, thr, keys=keys)
        rows.append(dict(model=name, conf=thr, subset=label,
                         n_boxes=sum(len(gtd[t]) for t in keys),
                         R=m['R'], TP=m['TP'], FN=m['FN']))
indep = pd.DataFrame(rows)
print(indep.to_string(index=False))
indep.to_csv(f'{DRIVE}/results_independence_threshold.csv', index=False)

Precision is omitted from the split table above: a false positive cannot be
attributed to one subset of the ground truth, so only recall is meaningful
when the ground truth is partitioned. Precision appears in the sweep and
operating-point tables, computed against the complete verified set.

In [ ]:
# Counting error at the same operating points
rows = []
for name, preds in preds_all.items():
    thr = float(best.loc[name, 'conf'])
    err, rel, signed = [], [], []
    for t in gt_all:
        pc = sum(1 for b in preds.get(t, []) if b[4] >= thr)
        gc = len(gt_all[t])
        err.append(abs(pc - gc)); signed.append(pc - gc)
        if gc: rel.append(abs(pc - gc) / gc)
    rows.append(dict(model=name, conf=thr,
                     MAE=round(float(np.mean(err)),2),
                     bias=round(float(np.mean(signed)),2),
                     rel_err_pct=round(float(np.mean(rel))*100,1)))
cnt = pd.DataFrame(rows)
print(cnt.to_string(index=False))
cnt.to_csv(f'{DRIVE}/results_counting_final.csv', index=False)